# 13 - E4 robustness degradations

**Question this notebook answers:** how much does each of ASVspoof2019 LA eval's detectors
degrade under four specific, realistic distribution shifts a deployed detector would meet —
lossy transcoding (MP3, Opus), additive background noise, and a lossy sample-rate round trip?

**Scope (decided by the user):** ONE severity level per degradation type, applied test-time
only to a fixed, seeded, stratified ~10,000-clip subset of the 2019 LA eval set (proportions
match the full eval set's label/attack_id mix):

| condition | what | why this level |
|---|---|---|
| `eval_mp3_32k` | MP3 (libmp3lame) CBR 32 kbps | common lossy "voice memo forwarded over messaging app" bitrate |
| `eval_opus_16k` | Opus (libopus) 16 kbps | VoIP-call-quality bitrate; Opus runs its own resampling internally |
| `eval_noise_10db` | MUSAN `noise/` mixed in at 10 dB SNR | audible-but-not-overwhelming background noise, the kind a phone mic actually picks up |
| `eval_resample_8k` | 16 -> 8 -> 16 kHz resample round trip | legacy telephony bandwidth (narrowband speech, no content above 4 kHz) |

Everything here is **test-time only** — no training data is degraded, no detector is
retrained on degraded audio. This notebook explains the generator
(`audio_df.degrade`, `scripts/make_degraded.py`); it does not score any detector (that is
A1/A2's job, on the manifests this generator writes).

Imports only from `audio_df.degrade` / `audio_df.data` / `audio_df.resources` and this
project's own scripts (`scripts/make_degraded.py`) — no feature extraction or model code is
defined here.

In [ ]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from audio_df.audio import load_audio  # noqa: E402 (after the scripts/ path setup)
from audio_df.degrade import degrade  # noqa: E402

_spec = importlib.util.spec_from_file_location(
    "make_degraded", REPO_ROOT / "scripts" / "make_degraded.py"
)
make_degraded = importlib.util.module_from_spec(_spec)
sys.modules["make_degraded"] = make_degraded
_spec.loader.exec_module(make_degraded)

CONFIG_PATH = REPO_ROOT / "configs" / "e4_degradations.yaml"
cfg = make_degraded.load_config(CONFIG_PATH)
print("seed:", cfg["seed"])
print("subset n:", cfg["subset"]["n"], " clip_ids stored:", len(cfg["subset"]["clip_ids"]))
print("conditions:", list(cfg["conditions"].keys()))

## Subset proportions vs the full eval set

`audio_df.data.stratified_subset` allocates the ~10,000 clips across `(label, attack_id)`
strata by the largest-remainder method, so the subset's class/attack mix should track the
full eval set's mix closely. **Mode:** if `data/manifests/eval.csv` exists (the real
ASVspoof 2019 LA eval manifest), this compares against it directly; otherwise a small
synthetic manifest stands in so this cell still runs.

In [ ]:
eval_manifest_path = REPO_ROOT / cfg["subset"]["source_manifest"]
if eval_manifest_path.exists():
    full_df = pd.read_csv(eval_manifest_path)
    mode = "real"
else:
    # Synthetic stand-in, same shape as the real manifest, so this cell is always reviewable.
    rows = []
    for i in range(200):
        rows.append(
            {
                "utt_id": f"SYN_{i:04d}",
                "label": "bonafide" if i < 30 else "spoof",
                "attack_id": "-" if i < 30 else f"A{7 + (i % 13):02d}",
            }
        )
    full_df = pd.DataFrame(rows)
    mode = "synthetic"
print("mode:", mode)

if mode == "real":
    sub_df = make_degraded.compute_subset(cfg, full_df)
else:
    # cfg["subset"]["clip_ids"] names real utt_ids that don't exist in our synthetic stand-in,
    # so recompute a fresh subset of the synthetic frame directly instead.
    from audio_df.data import stratified_subset

    sub_df = stratified_subset(full_df, n=60, strata=("label", "attack_id"), seed=cfg["seed"])

full_counts = full_df.groupby(["label", "attack_id"]).size()
sub_counts = sub_df.groupby(["label", "attack_id"]).size()
table = pd.DataFrame(
    {
        "full_n": full_counts,
        "full_pct": (full_counts / len(full_df) * 100).round(2),
        "subset_n": sub_counts,
        "subset_pct": (sub_counts / len(sub_df) * 100).round(2),
    }
).fillna(0)
table

## One clip under all four conditions

Pick one bonafide clip from the subset and run it through every condition, then compare
spectrograms. This is cheap (one clip, four ffmpeg subprocess round trips at most) — not a
full generation run. **Mode:** uses a real subset clip if the dataset is present locally,
else a synthesised 440 Hz tone standing in for "one clip" so the plotting code is still
exercised.

In [ ]:
SR = 16000

if eval_manifest_path.exists() and len(cfg["subset"]["clip_ids"]):
    row = full_df[full_df["utt_id"] == cfg["subset"]["clip_ids"][0]].iloc[0]
    x, sr = load_audio(row["path"], target_sr=SR)
    clip_id = row["utt_id"]
    print("using real clip:", clip_id, "duration_s=", len(x) / sr)
else:
    t = np.arange(int(1.5 * SR)) / SR
    x = (
        0.3 * np.sin(2 * np.pi * 440 * t) + 0.02 * np.random.default_rng(0).normal(size=len(t))
    ).astype(np.float32)
    sr = SR
    clip_id = "SYNTHETIC_DEMO_CLIP"
    print("dataset not found locally; using a synthetic 440 Hz tone as the demo clip")

# The noise condition needs a noise bank; use MUSAN if present, else a synthetic noise bank
# so this cell still runs end-to-end without the (un-downloaded, per this task's scope) MUSAN
# corpus.
noise_dir = REPO_ROOT / cfg["conditions"]["eval_noise_10db"]["noise_dir"]
if noise_dir.is_dir():
    noise_bank = make_degraded.load_noise_bank(noise_dir)
else:
    rng = np.random.default_rng(1)
    noise_bank = [rng.normal(0, 1, size=SR * 3).astype(np.float32) for _ in range(3)]
    print(
        "MUSAN noise/ not found locally; using synthetic white noise as a stand-in "
        "(run scripts/get_musan.py to fetch the real corpus)"
    )

outputs = {"clean": x}
for name in cfg["conditions"]:
    outputs[name] = degrade(
        x, sr, name, cfg, seed=cfg["seed"], clip_id=clip_id, noise_bank=noise_bank
    )

fig, axes = plt.subplots(len(outputs), 1, figsize=(9, 2.2 * len(outputs)), sharex=True)
for ax, (name, sig) in zip(axes, outputs.items(), strict=True):
    ax.specgram(sig, Fs=sr, NFFT=512, noverlap=256)
    ax.set_ylabel(name, fontsize=8)
fig.suptitle(f"clip {clip_id}: clean vs each E4 condition")
plt.tight_layout()
plt.show()

## Timing

`scripts/make_degraded.py timing [--n 200]` times all four conditions on the first N subset
clips and extrapolates to the full 4 x ~10,000-clip run (a **lower bound**, not a promise —
this is a fanless machine that throttles under sustained load). The cell below runs the same
`audio_df.resources.timing_run` machinery on a handful of synthetic clips just to show the
shape of the output; it is not a real timing estimate (see the terminal command below for
that).

In [ ]:
from audio_df.degrade import resample_roundtrip
from audio_df.resources import format_timing, timing_run

rng = np.random.default_rng(2)
demo_clips = [(rng.normal(0, 0.3, size=SR // 2)).astype(np.float32) for _ in range(5)]


def _resample_one(sig):
    return resample_roundtrip(sig, sr=SR, mid_sr=8000)


timing = timing_run(_resample_one, demo_clips, n=len(demo_clips), total_items=40000)
print(format_timing(timing))

## Exact terminal commands

(For the user to run — this notebook, and the agent that wrote it, do not run these.)

```bash
# 1. Fetch MUSAN's noise/ subset (11 GB tarball, ~1 GB kept). Dry-run first:
uv run python scripts/get_musan.py --dry-run
uv run python scripts/get_musan.py

# 2. Re-verify the subset clip list matches configs/e4_degradations.yaml (should already
#    match -- it was generated once, deterministically, from the real eval manifest):
uv run python scripts/make_degraded.py subset --verify

# 3. Time all four conditions on 200 clips before committing to the full run:
caffeinate -i uv run python scripts/make_degraded.py timing --n 200

# 4. Generate. caffeinate -i keeps the machine awake without forcing the display on;
#    resumable if interrupted (partial progress is never lost past the current chunk):
caffeinate -i uv run python scripts/make_degraded.py generate
# ...if interrupted, or if it exits 3 (MemoryPressureExit):
caffeinate -i uv run python scripts/make_degraded.py generate --resume

# 5. Spot-check the outputs:
uv run python scripts/make_degraded.py verify
```

**Disk/time estimates** (printed for real by step 3's `timing` command, and written to
`data/cache/timing/e4.json`): FLAC-encoded 16 kHz mono clips at ASVspoof 2019 LA's average
eval-clip duration (~4 s) run roughly 60-150 KB each; four conditions x ~10,000 clips is on
the order of a few GB total (`scripts/make_degraded.py`'s disk-estimate helper uses a
150 KB/clip conservative fallback when no timing run has been recorded yet, so it never
under-estimates). Wall time is dominated by the two codec conditions (ffmpeg subprocess
round trips); `noise` and `resample` are pure Python/`scipy` and much faster per clip.

## What comes next

Once `generate` has produced `data/manifests/eval_mp3_32k.csv`, `eval_opus_16k.csv`,
`eval_noise_10db.csv`, `eval_resample_8k.csv` (the manifest filename is exactly
`<condition_name>.csv` — every condition name already starts with `eval_`) and
`eval_clean10k.csv` (the clean reference: same clips, original paths):

```bash
# A1 (LFCC-B02+GMM) and A2 (SSL probe) score these five testsets the same way they score the
# ordinary eval set -- thresholds still come from dev, never from any of these:
uv run python -m audio_df.run_a1 score \
    --testsets eval_clean10k,eval_mp3_32k,eval_opus_16k,eval_noise_10db,eval_resample_8k
uv run python -m audio_df.run_a2 extract \
    --testsets eval_clean10k,eval_mp3_32k,eval_opus_16k,eval_noise_10db,eval_resample_8k
uv run python -m audio_df.run_a2 score \
    --testsets eval_clean10k,eval_mp3_32k,eval_opus_16k,eval_noise_10db,eval_resample_8k

# Then build the E4 table (per-condition EER/minDCF next to the clean10k reference):
uv run python scripts/make_tables.py --experiment e4
```

This notebook's job ends here: the generator is explained and demonstrated; scoring under
degradation is A1/A2's job, and turning those scores into the E4 comparison table is
`scripts/make_tables.py`'s job (both outside this task's ownership).